# Heart Disease Model Training Notebook

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os

CSV_PATH = os.environ.get("HEART_CSV", "heart (1).csv")
df = pd.read_csv(CSV_PATH)

ch_mean = df.loc[df["Cholesterol"] != 0, "Cholesterol"].mean()
df["Cholesterol"] = df["Cholesterol"].replace(0, ch_mean).round(2)

restingBP_mean = df.loc[df["RestingBP"] != 0, "RestingBP"].mean()
df["RestingBP"] = df["RestingBP"].replace(0, restingBP_mean).round(2)

df_encode = pd.get_dummies(df, drop_first=True)


X = df_encode.drop("HeartDisease", axis=1)
y = df_encode["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)

print(f"KNN  |  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")
print(classification_report(y_test, y_pred, target_names=["Low Risk", "High Risk"]))

joblib.dump(knn,            "model_01_KNN_heart.pkl")
joblib.dump(scaler,         "scaler.pkl")
joblib.dump(list(X.columns),"columns.pkl")
print("Saved: model_01_KNN_heart.pkl, scaler.pkl, columns.pkl")


KNN  |  Accuracy: 0.8533  |  F1: 0.8708
              precision    recall  f1-score   support

    Low Risk       0.80      0.86      0.83        77
   High Risk       0.89      0.85      0.87       107

    accuracy                           0.85       184
   macro avg       0.85      0.85      0.85       184
weighted avg       0.86      0.85      0.85       184

Saved: model_01_KNN_heart.pkl, scaler.pkl, columns.pkl
